In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets
# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
# TODO: Clusters on sales and/or holidays that occur together?


DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future
LAGS = [1, 2, 7, 30, 90, 182, 365]
DIFFS = [1, 30, 90, 182, 365]
ROLL_WINDOWS = { 7: 1, # window: lags
                30: [30, 90, 182, 365]}
HOLIDAY_WINDOWS = [-7, 0, 7]
TREND_WINDOW = 90
TREND_STEP = 30

In [ ]:
365 // 4

In [ ]:
df_train = pd.read_csv(TRAIN_FILE,
                       parse_dates=['Date'],
                       dtype={'StateHoliday': str} # 0 -> '0'
                       ).drop(['Customers'], axis=1)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0.
# This happens always. 
# We'll model without this and apply the rule at the end.
df_train.loc[df_train['Open'] == 0, 'Sales'] = np.nan
df_train['Sales'] = (df_train
                     .groupby(["Store"], group_keys=False)
                     .apply(lambda g: g
                            .set_index("Date")["Sales"]
                            .interpolate(method="time"),
                            include_groups=False)
                     .reset_index(level=0, drop=True))
df_train.drop(['Open'], axis=1, inplace=True)

In [ ]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)
df_train_store = attach_store_data(df_train, store_df)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store,
                            lags=LAGS,
                            roll_windows=ROLL_WINDOWS,
                            diffs=DIFFS,
                            holiday_windows=HOLIDAY_WINDOWS,
                            trends=[TREND_WINDOW, TREND_STEP])
#targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

In [ ]:
df_features.shape

In [ ]:
sales_df = df_train[['Date', 'Store', 'Sales']].copy()

promo_df = df_features[['Date', 'Store', 'Promo_counter']].copy()
promo2_df = df_features[['Date', 'Store', 'Promo2_counter']].copy()

df = pd.merge(sales_df, promo_df)
df = pd.merge(df, promo2_df)

In [ ]:
df_p = (pd.pivot(df, index='Date', columns='Store', values='Sales').sort_index())  # Sorted from oldest to newest
promo_p = (pd.pivot(df, index='Date', columns='Store', values='Promo_counter').sort_index())  # Sorted from oldest to newest
promo2_p = (pd.pivot(df, index='Date', columns='Store', values='Promo2_counter').sort_index())  # Sorted from oldest to newest

df_p.head()

In [ ]:
import numpy as np
import pandas as pd

def slope(df: pd.DataFrame) -> pd.Series:
    """
    Computes the linear trend (slope) for each column in a rolling window.

    Parameters
    ----------
    rolled : pd.DataFrame
        A windowed subset of the original DataFrame, as passed by rolling.apply().

    Returns
    -------
    pd.Series
        A Series of slope values, one per column.
    """
    if len(df) < 2:
        # Not enough data to compute slope
        return pd.Series(np.nan, index=df.columns)

    # Create numeric time vector for regression (0, 1, ..., n-1)
    X = np.arange(len(df))
    X_mean = X.mean()
    den = np.sum((X - X_mean) ** 2)

    # Vectorized slope computation: cov(X, Y) / var(X)
    slopes = ( (df - df.mean()).mul(X - X_mean, axis=0).sum(axis=0) ) / den

    return slopes

In [ ]:
df.columns[0], df.columns[1]

In [ ]:
df_p = (pd.pivot(df, index='Date', columns='Store', values='Sales').sort_index())  # Sorted from oldest to newest

slopes = df_p.rolling(window=TREND_WINDOW, min_periods=TREND_STEP, step=TREND_STEP).apply(slope)

In [ ]:
slopes.head(5)

In [ ]:
id_name = df_p.index.name
melt_index = [df_p.index.name, df_p.columns.name]
melt = lambda df, name: (df.reset_index()
                            .melt(id_vars=[id_name], value_name=name)
                            .set_index(melt_index))

melt(slopes, name = f'linear_trend')